> https://docs.langchain.com/oss/python/langgraph/add-memory

In [8]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

# 1. 모델 초기화
model = init_chat_model(
    "google_genai:gemini-3.1-flash-lite",
    temperature=0
)

In [ ]:
# uv add langgraph-checkpoint-sqlite
from typing import TypedDict, Annotated
import operator

from langchain_core.messages import AnyMessage
from langgraph.graph import StateGraph, START, END, MessagesState

# 2. State 정의
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

# 3. Node 정의
def llm_node(state: MessagesState):
    response = model.invoke(
        state["messages"]
    )
    return {"messages": [response]}

# 4. Graph 생성
graph_builder = StateGraph(MessagesState)

# 5. Graph에 Node 추가
graph_builder.add_node("llm", llm_node)

# 6. Edge 추가하여 Node 연결
graph_builder.add_edge(START, "llm")
graph_builder.add_edge("llm", END)

In [ ]:
# [sqlite 사용 방법]
# 1. `uv add "langgraph-checkpoint-sqlite<3.1.0"` -> 이 버전이 아닐 경우 오류 발생할 수 있음
# 2. `db` 폴더 생성
# 3. `https://sqlitebrowser.org/dl/` -> DB 정보 보기

# [sqlite 데이터 저장 방식]
# 기본적으로 상태 데이터를 JSON보다 빠르고 용량이 작은 바이너리 직렬화 포맷인 MessagePack 형식으로 변환
# 바이너리로 변환된 데이터는 SQLite 데이터베이스의 BLOB (Binary Large Object) 타입 컬럼에 저장

# [SQLite 파일]
# checkpoints.db (본체): 실제 데이터가 최종적으로 저장되는 메인 보관함
# checkpoints.db-wal (작업장): 성능 향상을 위해 변경 사항을 메인 파일에 옮기기 전 임시로 적어두는 노트
# checkpoints.db-shm (지도): 여러 프로그램이 동시에 접속할 때 작업장(wal)의 어디에 데이터가 있는지 알려주는 인덱스 지도

# from langgraph.checkpoint.memory import InMemorySaver
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver # InMemorySaver 대신 sqlite3과 SqliteSaver를 import 합니다.

# 7. Graph를 실행 가능한 형태로 컴파일
# checkpointer = InMemorySaver()
checkpointer = SqliteSaver(sqlite3.connect("./db/checkpoints.db", check_same_thread=False))
graph = graph_builder.compile(checkpointer=checkpointer)

# 8. Graph 실행
config = {"configurable": {"thread_id": "conversation_1"}}

In [13]:
from langchain_core.messages import HumanMessage

# 내 이름 알려주기
human_message = HumanMessage(content="안녕! 난 김일남이야.")
initial_message = {"messages": [human_message]}
graph.invoke(initial_message, config=config)

{'messages': [HumanMessage(content='안녕! 난 김일남이야.', additional_kwargs={}, response_metadata={}),
  AIMessage(content=[{'type': 'text', 'text': '안녕하세요, 일남 님! 만나서 정말 반가워요. 오늘 하루는 어떻게 보내고 계신가요? 궁금한 점이 있거나 도움이 필요하시면 언제든 편하게 말씀해 주세요!', 'extras': {'signature': 'EjQKMgEMOdbHyB6N9f4EIKuaC4Jp6FWd3qaxEuu5YYNWgwuFYLgJv7f8hbF7Ll4YujpqHRpH'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019eba76-76ef-7631-a557-b734923488d5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 43, 'total_tokens': 53, 'input_token_details': {'cache_read': 0}})]}

In [14]:
# 내 이름 물어보기
human_message = HumanMessage(content="내 이름이 뭐야?")
initial_message = {"messages": [human_message]}
result = graph.invoke(initial_message, config=config)
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

안녕! 난 김일남이야.
================================== Ai Message ==================================

[{'type': 'text', 'text': '안녕하세요, 일남 님! 만나서 정말 반가워요. 오늘 하루는 어떻게 보내고 계신가요? 궁금한 점이 있거나 도움이 필요하시면 언제든 편하게 말씀해 주세요!', 'extras': {'signature': 'EjQKMgEMOdbHyB6N9f4EIKuaC4Jp6FWd3qaxEuu5YYNWgwuFYLgJv7f8hbF7Ll4YujpqHRpH'}}]
================================ Human Message =================================

내 이름이 뭐야?
================================== Ai Message ==================================

[{'type': 'text', 'text': '방금 말씀해 주셨잖아요! 당신의 이름은 **김일남** 님입니다. :)', 'extras': {'signature': 'EjQKMgEMOdbHZx9GL2zKhW47GJYIRozZowkbpEtbFuEAke4DwSGc+9pRge/Re/Lvbir5avcx'}}]


In [15]:
config = {"configurable": {"thread_id": "conversation_2"}}

In [16]:
# 내 이름 물어보기
human_message = HumanMessage(content="내 이름이 뭐야?")
initial_message = {"messages": [human_message]}
result = graph.invoke(initial_message, config=config)
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

내 이름이 뭐야?
================================== Ai Message ==================================

[{'type': 'text', 'text': '죄송하지만, 저는 사용자의 개인정보를 저장하거나 기억하지 않기 때문에 당신의 이름을 알지 못합니다. \n\n혹시 이전에 저에게 이름을 알려주셨더라도, 새로운 대화 세션이 시작되면 이전 대화의 내용을 기억할 수 없습니다. 원하신다면 지금 이름을 알려주세요! 기억해 두겠습니다.', 'extras': {'signature': 'EjQKMgEMOdbHou6VdYldovTec2GhrtIIEAzgOxPSNnbqJhUOfGX0todDNeX+6tzzk6VUu9Oi'}}]
